# Convert local dataset to HuggingFace repo

In [47]:
import pandas as pd
import numpy as np

from huggingface_hub import login, whoami
from datasets import Dataset, DatasetDict
from pathlib import Path

Log into HuggingFace via access token

In [48]:
HF_TOKEN="<HUGGING_FACE_TOKEN_WITH_PERMISSION_TO_CREATE_DATASET>"
login(token=HF_TOKEN)

print(whoami())

{'type': 'user', 'id': '6aad11231339b5a189922a50', 'name': 'choisongjun', 'fullname': 'HT', 'isPro': False, 'avatarUrl': '/avatars/481c7e7c8d0b9c678adf6db8ab7a351d.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'MedGemmaToken-1', 'role': 'fineGrained', 'createdAt': '2026-09-19T06:17:45.673Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6aad11231339b5a189922a50', 'type': 'user', 'name': 'choisongjun'}, 'permissions': ['repo.content.read']}]}}}}


Locate custom dataset

In [49]:
def find_dataset_root(dirname: str = "<DATASET_NAME>", search_root: Path = None) -> Path:
    root = search_root or Path.cwd()

    direct = root / dirname
    if direct.is_dir():
        return direct

    matches = [p for p in root.rglob(dirname) if p.is_dir()]
    if matches:
        return matches[0]

    parent = root
    searched = [str(root)]
    for _ in range(4):
        parent = parent.parent
        searched.append(str(parent))
        matches = [p for p in parent.rglob(dirname) if p.is_dir()]
        if matches:
            return matches[0]

    raise FileNotFoundError(f"Could not find '{dirname}'. Searched under: {searched}")

DATASET_ROOT = find_dataset_root()
print("Using:", DATASET_ROOT)

Using: /home/tokuden/VGU_WS26_ClinicalProject_BHTBH/BHTBH/extracted_dataset_1


Identify all diseases in customized dataset:

In [42]:
def get_disease_folders(dataset_root: Path) -> list[str]:
    return sorted([p.name for p in dataset_root.iterdir() if p.is_dir()])

Walk `extracted_dataset_1` for each disease folder, and build one combined table with columns `case_id, case_text, disease`.

In [43]:
DISEASES = get_disease_folders(DATASET_ROOT)

all_rows = []

for disease in DISEASES:
    csv_path = DATASET_ROOT / disease / "cases_with_image.csv"

    if not csv_path.is_file():
        continue

    disease_df = pd.read_csv(csv_path, dtype=str)

    rows = pd.DataFrame({
        "case_id": disease_df["case_id"],
        "case_text": disease_df["case_text"],
        "disease": disease,
    })
    all_rows.append(rows)

all_cases_df = pd.concat(all_rows, ignore_index=True)

print(f"\nTotal combined: {len(all_cases_df)} rows across {all_cases_df['disease'].nunique()} diseases")
all_cases_df.head()


Total combined: 98 rows across 8 diseases


,case_id,case_text,disease
0,PMC8183367_01,"A 38-year-old man, forest guard, resident of a...",dengue
1,PMC4327404_01,A 54-year-old female was referred to our depar...,dengue
2,PMC4046007_01,A previously healthy 55-year-old man with no p...,dengue
3,PMC5288990_01,A 45-year-old man presented with a history of ...,dengue
4,PMC5566360_01,A 17-year-old male presented to the hospital e...,dengue


General function that takes care of splitting the total dataset and returning train + test set. Comes with a default algorithm which can be changed based on the function passed in.

In [44]:
def separate(df: pd.DataFrame, group_col: str = "disease", split_algorithm=None, seed: int = 42):
    if split_algorithm is None:
        split_algorithm = lambda test_proportion: 0.20        # default: always 20% test, regardless of group size

    rng = np.random.default_rng(seed)                         # default: rng seed 42
    train_parts, test_parts = [], []

    for group_val, group in df.groupby(group_col):
        n = len(group)
        test_proportion = split_algorithm(n)
        n_test = round(n * test_proportion)

        shuffled_idx = rng.permutation(group.index.to_numpy())
        test_idx = shuffled_idx[:n_test]
        train_idx = shuffled_idx[n_test:]

        test_parts.append(group.loc[test_idx])
        train_parts.append(group.loc[train_idx])

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, test_df

Separate dataset into train and test set, then upload both to HuggingFace repo

In [45]:
def test_fraction_for(n: int) -> float:
    """Tiered test-set fraction based on how many cases a disease has."""
    if n >= 20:
        return 0.20
    elif n >= 5:  # 5 <= n < 20
        return 0.10
    else:  # n < 5
        return 0.0

train_df, test_df = separate(df=all_cases_df, split_algorithm=test_fraction_for, seed=69420)

print(f"\nTotal: train={len(train_df)} | test={len(test_df)}")


Total: train=86 | test=12


In [46]:
REPO_ID = "<REPO_USER>/<REPO_NAME>"
dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

dataset_dict.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f"\nPushed to https://huggingface.co/datasets/{REPO_ID}")

HfHubHTTPError: (Request ID: Root=1-6abb61f2-28f9df9e6d5a68157f8ca7db;8d0697b5-6b9a-4da1-a716-532663cceb8d)

403 Forbidden: You don't have the rights to create a dataset under the namespace "Tokuden".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.